# 09 — Landmarks, particiones y características temporales

Auditoría reproducible sobre MIMIC-IV Demo v2.2. Este notebook abre exclusivamente `development` y `validation`: la partición `test` no se descubre, carga ni resume. Todas las salidas son agregadas y no deben interpretarse como resultados clínicos.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys, tempfile
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import SVG, display
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src')) if str(PROJECT_ROOT / 'src') not in sys.path else None
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None
from mimic_sepsis.artifacts import ArtifactStore, ArtifactValidationError
from mimic_sepsis.features import numeric_window_features
from scripts.build_demo_sofa_incremental import canonical_config
config = canonical_config()
candidates = sorted((PROJECT_ROOT / 'data/derived/sofa').glob('*/60_features/sepsis3_development_features.manifest.json'))
valid = []
for manifest_path in candidates:
    store = ArtifactStore(manifest_path.parent)
    try:
        manifest = store.validate('sepsis3_development_features', expected_config=config)
        valid.append((manifest.created_at_utc, manifest_path.parents[1]))
    except (FileNotFoundError, ArtifactValidationError):
        pass
if not valid:
    raise RuntimeError('No hay artefactos compatibles. Ejecute: python scripts/build_demo_sofa_incremental.py --stage all --resume')
RUN_ROOT = sorted(valid, key=lambda item: (item[0], str(item[1])))[-1][1]
landmark_store = ArtifactStore(RUN_ROOT / '50_landmarks')
feature_store = ArtifactStore(RUN_ROOT / '60_features')
print(f'Ejecución validada: {RUN_ROOT.name}')

## Integridad de las matrices y cobertura

In [ ]:
partitions = ('development', 'validation')
targets = ('sepsis3', 'septic_shock')
variables = tuple(config['features']['variables'])
outcome_rows, coverage_rows = [], []
for target in targets:
    for partition in partitions:
        stem = f'{target}_{partition}'
        landmarks = landmark_store.read_dataframe(f'{stem}_landmarks', expected_config=config)
        features = feature_store.read_dataframe(f'{stem}_features', expected_config=config)
        keys = ['subject_id', 'hadm_id', 'stay_id', 'landmark_time']
        assert not features.duplicated(['stay_id', 'landmark_time']).any()
        assert not any(column.startswith('outcome') for column in features)
        joined = landmarks.merge(features, on=keys, how='left', validate='many_to_one')
        assert joined.filter(regex='_count_').notna().all(axis=None)
        for horizon, frame in joined.groupby('horizon_hours'):
            observed = frame['horizon_observed']
            outcome_rows.append({'target': target, 'partition': partition, 'horizon_hours': int(horizon), 'landmarks': len(frame), 'observed': int(observed.sum()), 'positive': int(frame.loc[observed, 'outcome'].sum())})
        for variable in variables:
            coverage_rows.append({'target': target, 'partition': partition, 'variable': variable, 'window_hours': 24, 'landmarks': len(features), 'observed': int((~features[f'{variable}_missing_24h']).sum())})
outcome_summary = pd.DataFrame(outcome_rows)
coverage_summary = pd.DataFrame(coverage_rows)
coverage_summary['percent_observed'] = 100 * coverage_summary['observed'] / coverage_summary['landmarks']
display(outcome_summary, coverage_summary)

## Control sintético contra fuga temporal

In [ ]:
L = pd.Timestamp('2100-01-02 00:00')
point = pd.DataFrame({'subject_id':[1], 'hadm_id':[10], 'stay_id':[100], 'landmark_time':[L]})
before = pd.DataFrame({'stay_id':[100], 'event_time':[L-pd.Timedelta(hours=1)], 'value':[80.]})
with_future = pd.concat([before, pd.DataFrame({'stay_id':[100], 'event_time':[L+pd.Timedelta(seconds=1)], 'value':[999.]})], ignore_index=True)
left = numeric_window_features(point, before, variable='heart_rate')
right = numeric_window_features(point, with_future, variable='heart_rate')
pd.testing.assert_frame_equal(left, right)
print('OK: un evento posterior al landmark no modifica ningún predictor.')

## Misma cobertura, dos gramáticas gráficas

In [ ]:
plot_data = coverage_summary.query("target == 'sepsis3' and partition == 'development'").sort_values('percent_observed')
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(plot_data['variable'], plot_data['percent_observed'], color='#2878B5')
ax.set(xlabel='Landmarks con medición en [L-24 h, L) (%)', ylabel='', title='Cobertura de predictores — desarrollo Sepsis-3')
ax.set_xlim(0, 100); fig.tight_layout(); plt.show()
rscript = shutil.which('Rscript')
if not rscript: raise RuntimeError('Rscript no está disponible en el entorno.')
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp); csv = tmp/'coverage.csv'; svg = tmp/'coverage.svg'; script = tmp/'plot.R'
    plot_data.to_csv(csv, index=False)
    script.write_text("""args <- commandArgs(trailingOnly=TRUE)
suppressPackageStartupMessages(library(ggplot2))
d <- read.csv(args[1])
p <- ggplot(d, aes(percent_observed, reorder(variable, percent_observed))) + geom_col(fill='#2878B5') + scale_x_continuous(limits=c(0,100)) + labs(title='Cobertura de predictores — desarrollo Sepsis-3', x='Landmarks con medición en [L-24 h, L) (%)', y=NULL) + theme_minimal(base_size=12)
ggsave(args[2], p, width=8, height=4.8, device=grDevices::svg)
""")
    subprocess.run([rscript, str(script), str(csv), str(svg)], check=True, capture_output=True, text=True)
    display(SVG(filename=str(svg)))

## Criterio para avanzar

El bloque queda listo para modelado cuando los manifiestos validan, no hay claves duplicadas ni columnas de desenlace en las matrices, el test permanece sin abrir y la cobertura se interpreta con sus indicadores de ausencia. El demo comprueba ingeniería; no tiene tamaño para estimar rendimiento clínico.